## LOAD DATASET FROM AZURE SQL

In [ ]:
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from us_visa.configuration.azure_config import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM [EasyVisa]", engine)

print("Loaded from: Azure SQL table EasyVisa")
print("Shape:", df.shape)
df.head()

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (939273398.py, line 1)

## Feature Engineering / Data Pre-processing

### Outlier Check

In [ ]:
# Outlier detection with boxplots for numeric columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(15, 10))
for i, variable in enumerate(numeric_columns, start=1):
    plt.subplot(2, 2, i)
    plt.boxplot(df[variable], whis=1.5)
    plt.title(variable)
    plt.tight_layout()

plt.show()

NameError: name 'df' is not defined

In [ ]:
# --- Data Preprocessing and Feature Engineering ---
data = df.copy()

# 1) Data quality fix
if "no_of_employees" in data.columns:
    data["no_of_employees"] = data["no_of_employees"].abs()

# 2) Drop ID column
data.drop(columns=["case_id"], errors="ignore", inplace=True)

# 3) Build annualized wage feature
wage_factor = {"Hour": 2080, "Week": 52, "Month": 12, "Year": 1}
if {"prevailing_wage", "unit_of_wage"}.issubset(data.columns):
    data["annualized_wage"] = data["prevailing_wage"] * data["unit_of_wage"].map(wage_factor).fillna(1)

# 4) Encode target
data["case_status"] = data["case_status"].map({"Certified": 1, "Denied": 0})

# 5) Encode Y/N binary columns
binary_cols = ["has_job_experience", "requires_job_training", "full_time_position"]
for col in binary_cols:
    if col in data.columns:
        data[col] = data[col].map({"Y": 1, "N": 0})

# 6) One-hot encode remaining categorical columns
X = data.drop(columns=["case_status"])
y = data["case_status"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 7) Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1, stratify=y
)

# 8) Scale numeric features for model experiments
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature engineering complete")
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Target balance (train):")
print(y_train.value_counts(normalize=True))